# AI SQL Analyst Agent

This notebook builds an LLM-powered SQL analyst agent that converts natural language business questions into SQL queries, executes them against a relational database, validates the results, and returns business-friendly answers.

In [1]:
import sqlite3
import pandas as pd

In [3]:
db_path = "/Users/rheageorge/Downloads/Chinook_Sqlite.sqlite"

In [7]:
conn = sqlite3.connect(db_path)

In [9]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


View sample data from important tables

In [11]:
important_tables = [
    "Customer",
    "Invoice",
    "InvoiceLine",
    "Track",
    "Album",
    "Artist",
    "Genre",
    "Employee"
]

for table in important_tables:
    print(f"\n--- {table} ---")
    display(pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5;", conn))


--- Customer ---


,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
1,2,Leonie,Köhler,None,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,+49 0711 2842222,None,leonekohler@surfeu.de,5
2,3,François,Tremblay,None,1498 rue Bélanger,Montréal,QC,Canada,H2G 1A7,+1 (514) 721-4711,None,ftremblay@gmail.com,3
3,4,Bjørn,Hansen,None,Ullevålsveien 14,Oslo,None,Norway,0171,+47 22 44 22 22,None,bjorn.hansen@yahoo.no,4
4,5,František,Wichterlová,JetBrains s.r.o.,Klanova 9/506,Prague,None,Czech Republic,14700,+420 2 4172 5555,+420 2 4172 5555,frantisekw@jetbrains.com,4



--- Invoice ---


,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2009-01-01 00:00:00,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,2,4,2009-01-02 00:00:00,Ullevålsveien 14,Oslo,None,Norway,0171,3.96
2,3,8,2009-01-03 00:00:00,Grétrystraat 63,Brussels,None,Belgium,1000,5.94
3,4,14,2009-01-06 00:00:00,8210 111 ST NW,Edmonton,AB,Canada,T6G 2C7,8.91
4,5,23,2009-01-11 00:00:00,69 Salem Street,Boston,MA,USA,2113,13.86



--- InvoiceLine ---


...

,ArtistId,Name
0,1,AC/DC
1,2,Accept
2,3,Aerosmith
3,4,Alanis Morissette
4,5,Alice In Chains



--- Genre ---


,GenreId,Name
0,1,Rock
1,2,Jazz
2,3,Metal
3,4,Alternative & Punk
4,5,Rock And Roll



--- Employee ---


,EmployeeId,LastName,FirstName,Title,ReportsTo,BirthDate,HireDate,Address,City,State,Country,PostalCode,Phone,Fax,Email
0,1,Adams,Andrew,General Manager,NaN,1962-02-18 00:00:00,2002-08-14 00:00:00,11120 Jasper Ave NW,Edmonton,AB,Canada,T5K 2N1,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com
1,2,Edwards,Nancy,Sales Manager,1.0,1958-12-08 00:00:00,2002-05-01 00:00:00,825 8 Ave SW,Calgary,AB,Canada,T2P 2T3,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com
2,3,Peacock,Jane,Sales Support Agent,2.0,1973-08-29 00:00:00,2002-04-01 00:00:00,1111 6 Ave SW,Calgary,AB,Canada,T2P 5M5,+1 (403) 262-3443,+1 (403) 262-6712,jane@chinookcorp.com
3,4,Park,Margaret,Sales Support Agent,2.0,1947-09-19 00:00:00,2003-05-03 00:00:00,683 10 Street SW,Calgary,AB,Canada,T2P 5G3,+1 (403) 263-4423,+1 (403) 263-4289,margaret@chinookcorp.com
4,5,Johnson,Steve,Sales Support Agent,2.0,1965-03-03 00:00:00,2003-10-17 00:00:00,7727B 41 Ave,Calgary,AB,Canada,T3B 1Y7,1 (780) 836-9987,1 (780) 836-9543,steve@chinookcorp.com


Get table schemas

In [13]:
def get_table_schema(table_name):
    query = f"PRAGMA table_info({table_name});"
    return pd.read_sql_query(query, conn)

for table in important_tables:
    print(f"\n--- Schema for {table} ---")
    display(get_table_schema(table))


--- Schema for Customer ---


,cid,name,type,notnull,dflt_value,pk
0,0,CustomerId,INTEGER,1,None,1
1,1,FirstName,NVARCHAR(40),1,None,0
2,2,LastName,NVARCHAR(20),1,None,0
3,3,Company,NVARCHAR(80),0,None,0
4,4,Address,NVARCHAR(70),0,None,0
5,5,City,NVARCHAR(40),0,None,0
6,6,State,NVARCHAR(40),0,None,0
7,7,Country,NVARCHAR(40),0,None,0
8,8,PostalCode,NVARCHAR(10),0,None,0
9,9,Phone,NVARCHAR(24),0,None,0



--- Schema for Invoice ---


,cid,name,type,notnull,dflt_value,pk
0,0,InvoiceId,INTEGER,1,None,1
1,1,CustomerId,INTEGER,1,None,0
2,2,InvoiceDate,DATETIME,1,None,0
3,3,BillingAddress,NVARCHAR(70),0,None,0
4,4,BillingCity,NVARCHAR(40),0,None,0
5,5,BillingState,NVARCHAR(40),0,None,0
6,6,BillingCountry,NVARCHAR(40),0,None,0
7,7,BillingPostalCode,NVARCHAR(10),0,None,0
8,8,Total,"NUMERIC(10,2)",1,None,0



--- Schema for InvoiceLine ---


...

,cid,name,type,notnull,dflt_value,pk
0,0,ArtistId,INTEGER,1,None,1
1,1,Name,NVARCHAR(120),0,None,0



--- Schema for Genre ---


,cid,name,type,notnull,dflt_value,pk
0,0,GenreId,INTEGER,1,None,1
1,1,Name,NVARCHAR(120),0,None,0



--- Schema for Employee ---


,cid,name,type,notnull,dflt_value,pk
0,0,EmployeeId,INTEGER,1,None,1
1,1,LastName,NVARCHAR(20),1,None,0
2,2,FirstName,NVARCHAR(20),1,None,0
3,3,Title,NVARCHAR(30),0,None,0
4,4,ReportsTo,INTEGER,0,None,0
5,5,BirthDate,DATETIME,0,None,0
6,6,HireDate,DATETIME,0,None,0
7,7,Address,NVARCHAR(70),0,None,0
8,8,City,NVARCHAR(40),0,None,0
9,9,State,NVARCHAR(40),0,None,0


Create a readable schema summary

In [15]:
def get_database_schema(conn):
    tables_df = pd.read_sql_query(
        """
        SELECT name 
        FROM sqlite_master 
        WHERE type = 'table'
        ORDER BY name;
        """,
        conn
    )

    schema_text = ""

    for table_name in tables_df["name"]:
        columns_df = pd.read_sql_query(f"PRAGMA table_info({table_name});", conn)

        schema_text += f"\nTable: {table_name}\n"
        schema_text += "Columns:\n"

        for _, row in columns_df.iterrows():
            schema_text += f" - {row['name']} ({row['type']})\n"

    return schema_text

schema_text = get_database_schema(conn)
print(schema_text)


Table: Album
Columns:
 - AlbumId (INTEGER)
 - Title (NVARCHAR(160))
 - ArtistId (INTEGER)

Table: Artist
Columns:
 - ArtistId (INTEGER)
 - Name (NVARCHAR(120))

Table: Customer
Columns:
 - CustomerId (INTEGER)
 - FirstName (NVARCHAR(40))
 - LastName (NVARCHAR(20))
 - Company (NVARCHAR(80))
 - Address (NVARCHAR(70))
 - City (NVARCHAR(40))
 - State (NVARCHAR(40))
 - Country (NVARCHAR(40))
 - PostalCode (NVARCHAR(10))
 - Phone (NVARCHAR(24))
 - Fax (NVARCHAR(24))
 - Email (NVARCHAR(60))
 - SupportRepId (INTEGER)

Table: Employee
Columns:
 - EmployeeId (INTEGER)
 - LastName (NVARCHAR(20))
 - FirstName (NVARCHAR(20))
 - Title (NVARCHAR(30))
 - ReportsTo (INTEGER)
 - BirthDate (DATETIME)
 - HireDate (DATETIME)
 - Address (NVARCHAR(70))
 - City (NVARCHAR(40))
 - State (NVARCHAR(40))
 - Country (NVARCHAR(40))
 - PostalCode (NVARCHAR(10))
 - Phone (NVARCHAR(24))
 - Fax (NVARCHAR(24))
 - Email (NVARCHAR(60))

Table: Genre
Columns:
 - GenreId (INTEGER)
 - Name (NVARCHAR(120))

Table: Invoice
Col

Testing one manual SQL query

In [17]:
query = """
SELECT 
    c.CustomerId,
    c.FirstName || ' ' || c.LastName AS customer_name,
    c.Country,
    ROUND(SUM(i.Total), 2) AS total_revenue
FROM Customer c
JOIN Invoice i
    ON c.CustomerId = i.CustomerId
GROUP BY 
    c.CustomerId,
    customer_name,
    c.Country
ORDER BY total_revenue DESC
LIMIT 10;
"""

top_customers = pd.read_sql_query(query, conn)
top_customers

,CustomerId,customer_name,Country,total_revenue
0,6,Helena Holý,Czech Republic,49.62
1,26,Richard Cunningham,USA,47.62
2,57,Luis Rojas,Chile,46.62
3,45,Ladislav Kovács,Hungary,45.62
4,46,Hugh O'Reilly,Ireland,45.62
5,24,Frank Ralston,USA,43.62
6,28,Julia Barnett,USA,43.62
7,37,Fynn Zimmermann,Germany,43.62
8,7,Astrid Gruber,Austria,42.62
9,25,Victor Stevens,USA,42.62


Createing a reusable SQL execution function

In [19]:
def execute_sql_query(query, conn=conn):
    """
    Executes a SQL query against the Chinook SQLite database.
    Returns either a result dataframe or an error message.
    """
    try:
        result = pd.read_sql_query(query, conn)
        return {
            "success": True,
            "data": result,
            "error": None
        }
    except Exception as e:
        return {
            "success": False,
            "data": None,
            "error": str(e)
        }

In [21]:
test_result = execute_sql_query(query)

if test_result["success"]:
    display(test_result["data"])
else:
    print(test_result["error"])

,CustomerId,customer_name,Country,total_revenue
0,6,Helena Holý,Czech Republic,49.62
1,26,Richard Cunningham,USA,47.62
2,57,Luis Rojas,Chile,46.62
3,45,Ladislav Kovács,Hungary,45.62
4,46,Hugh O'Reilly,Ireland,45.62
5,24,Frank Ralston,USA,43.62
6,28,Julia Barnett,USA,43.62
7,37,Fynn Zimmermann,Germany,43.62
8,7,Astrid Gruber,Austria,42.62
9,25,Victor Stevens,USA,42.62


In [23]:
bad_query = """
SELECT customer_name, total_sales
FROM Customers;
"""

bad_result = execute_sql_query(bad_query)

bad_result

{'success': False,
 'data': None,
 'error': "Execution failed on sql '\nSELECT customer_name, total_sales\nFROM Customers;\n': no such table: Customers"}

In [25]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key:  ········


In [27]:
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully.")
else:
    print("API key not found.")

API key loaded successfully.


Create the AutoGen LLM configuration

import sys
!{sys.executable} -m pip install pyautogen==0.2.35

In [33]:
import autogen
print("AutoGen imported successfully")

/opt/anaconda3/lib/python3.12/site-packages/flaml/__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


AutoGen imported successfully


In [35]:
import autogen
import os

config_list = [
    {
        "model": "gpt-4o-mini",
        "api_key": os.getenv("OPENAI_API_KEY")
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0
}

print("AutoGen LLM config created successfully.")

AutoGen LLM config created successfully.


Create the SQL Analyst Agent

In [39]:
sql_analyst = autogen.AssistantAgent(
    name="SQL_Analyst_Agent",
    llm_config=llm_config,
    system_message=f"""
You are an expert SQL analyst.

Your job is to answer business questions by writing SQLite SQL queries.

Use only the tables and columns provided in the database schema below.
Do not invent table names or column names.

Database schema:
{schema_text}

Rules:
1. Write valid SQLite SQL only.
2. Use joins when needed.
3. Use clear aliases for calculated fields.
4. Limit result sets to 10 rows unless the user asks otherwise.
5. After writing SQL, explain what the query is doing in simple business terms.
6. If a SQL query fails, use the error message to correct the query.
"""
)

print("SQL Analyst Agent created successfully.")

[autogen.oai.client: 06-01 12:10:35] {164} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
SQL Analyst Agent created successfully.


Create a User Proxy Agent

In [41]:
user_proxy = autogen.UserProxyAgent(
    name="User_Proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    code_execution_config=False
)

print("User Proxy Agent created successfully.")

User Proxy Agent created successfully.


In [43]:
business_question = "Who are the top 5 customers by total revenue?"

user_proxy.initiate_chat(
    sql_analyst,
    message=f"""
Answer this business question using the database schema provided:

{business_question}

Return:
1. The SQL query
2. A short explanation of the logic
"""
)

User_Proxy (to SQL_Analyst_Agent):


Answer this business question using the database schema provided:

Who are the top 5 customers by total revenue?

Return:
1. The SQL query
2. A short explanation of the logic


--------------------------------------------------------------------------------


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

Extract SQL from the agent response

In [ ]:
import re

def extract_sql_from_response(response_text):
    """
    Extracts SQL code from a markdown SQL code block.
    If no code block is found, returns the full response text.
    """
    sql_pattern = r"```sql\s*(.*?)```"
    match = re.search(sql_pattern, response_text, re.DOTALL | re.IGNORECASE)

    if match:
        return match.group(1).strip()

    return response_text.strip()

Create a full question-to-SQL execution function

In [ ]:
def ask_sql_agent(question):
    """
    Sends a natural language business question to the AutoGen SQL analyst agent,
    extracts the generated SQL query, executes it against SQLite,
    and returns the SQL plus query result.
    """

    chat_result = user_proxy.initiate_chat(
        sql_analyst,
        message=f"""
Business question:
{question}

Generate a SQLite SQL query to answer this question.
Return the SQL in a ```sql code block and explain the logic briefly.
"""
    )

    # Get the last assistant message
    messages = chat_result.chat_history
    assistant_messages = [
        msg["content"] for msg in messages 
        if msg.get("name") == "SQL_Analyst_Agent" or msg.get("role") == "assistant"
    ]

    response_text = assistant_messages[-1]
    generated_sql = extract_sql_from_response(response_text)

    result = execute_sql_query(generated_sql)

    return {
        "question": question,
        "agent_response": response_text,
        "generated_sql": generated_sql,
        "execution_result": result
    }

In [ ]:
response = ask_sql_agent("Who are the top 5 customers by total revenue?")

print("Question:")
print(response["question"])

print("\nGenerated SQL:")
print(response["generated_sql"])

if response["execution_result"]["success"]:
    print("\nQuery Result:")
    display(response["execution_result"]["data"])
else:
    print("\nSQL Error:")
    print(response["execution_result"]["error"])

In [ ]:
def ask_sql_agent_with_validation(question, max_retries=2):
    """
    Sends a business question to the AutoGen SQL Analyst Agent,
    extracts SQL, executes it, and retries with error feedback if the SQL fails.
    """

    prompt = f"""
Business question:
{question}

Generate a valid SQLite SQL query to answer this question.

Return:
1. SQL query inside a ```sql code block
2. A brief explanation of the business logic
"""

    for attempt in range(max_retries + 1):
        chat_result = user_proxy.initiate_chat(
            sql_analyst,
            message=prompt
        )

        messages = chat_result.chat_history

        assistant_messages = [
            msg["content"] for msg in messages
            if msg.get("name") == "SQL_Analyst_Agent" or msg.get("role") == "assistant"
        ]

        response_text = assistant_messages[-1]
        generated_sql = extract_sql_from_response(response_text)

        execution_result = execute_sql_query(generated_sql)

        if execution_result["success"]:
            return {
                "question": question,
                "success": True,
                "attempts": attempt + 1,
                "agent_response": response_text,
                "generated_sql": generated_sql,
                "data": execution_result["data"],
                "error": None
            }

        prompt = f"""
The previous SQL query failed.

Business question:
{question}

Failed SQL query:
```sql
{generated_sql}

In [ ]:
sample_questions = [
    "Who are the top 5 customers by total revenue?",
    "Which countries generate the most revenue?",
    "What are the top 5 selling genres?",
    "Which sales support agent manages the highest revenue customers?",
    "Which artists generated the most sales?"
]